In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import pandas as pd

from turbofan.config.schema import load_config
from turbofan.data.loader import load_raw_train, load_raw_test, load_rul_labels
from turbofan.eda import quality, sensors, degradation

pio.templates.default = "plotly_white"
plt.rcParams["figure.figsize"] = (12, 6)

if Path.cwd().name == "notebooks":
    %cd ..

cfg = load_config(Path("configs/default.yaml"))
cfg = cfg.model_copy(
    update={"data": cfg.data.model_copy(update={"fd_subset": "FD003"})}
)
print(cfg.data.fd_subset)

In [ ]:
train_df = load_raw_train(cfg.data)
test_df  = load_raw_test(cfg.data)
test_rul = load_rul_labels(cfg.data)

print(f"Train: {train_df.shape[0]:,} rows, {train_df['engine_id'].nunique()} engines")
print(f"Test:  {test_df.shape[0]:,} rows,  {test_df['engine_id'].nunique()} engines")
print(f"RUL labels: {len(test_rul)} engines")
train_df.head()

## 1. Data Quality

In [ ]:
missing = quality.find_missing_values(train_df)
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "None — dataset is complete.")

In [ ]:
constant = quality.find_constant_sensors(train_df)
sensor_cols = [c for c in train_df.columns if c.startswith("s_")]
non_constant = [c for c in sensor_cols if c not in constant]

print(f"Constant sensors ({len(constant)}): {constant}")
print(f"Non-constant sensors ({len(non_constant)}): {non_constant}")

## 2. Operational Settings

In [ ]:
op_cols = ["op_1", "op_2", "op_3"]

op_combos = (
    train_df[op_cols]
    .round(0)
    .groupby(op_cols)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(f"Operating condition combinations: {len(op_combos)}")
op_combos

## 3. RUL Labels

In [ ]:
train_with_rul = degradation.compute_rul_curves(train_df, max_rul=cfg.data.max_rul)
rul = train_with_rul["rul"]

lifetimes = train_df.groupby("engine_id")["cycle"].max()
print(f"Engine lifetime — min: {lifetimes.min()}, median: {lifetimes.median():.0f}, max: {lifetimes.max()}")

fig = px.histogram(
    lifetimes,
    nbins=30,
    labels={"value": "Max cycle", "count": "Engines"},
    title="Engine Lifetime Distribution",
)
fig.show()

## 4. Sensor Stats

In [ ]:
stats = sensors.compute_sensor_stats(train_df)
stats.loc[non_constant].round(3)

## 5. Correlation Filter

In [ ]:
CORR_THRESHOLD = 0.1

rul_corr = train_df[non_constant].corrwith(rul).sort_values()

fig = px.bar(
    x=rul_corr.values,
    y=rul_corr.index,
    orientation="h",
    labels={"x": "Pearson r with RUL", "y": "Sensor"},
    title=f"Sensor-RUL Correlation (threshold +/-{CORR_THRESHOLD})",
)
fig.add_vline(x=CORR_THRESHOLD,  line_dash="dash", line_color="red")
fig.add_vline(x=-CORR_THRESHOLD, line_dash="dash", line_color="red")
fig.show()

informative = rul_corr[rul_corr.abs() >= CORR_THRESHOLD].index.tolist()
dropped     = rul_corr[rul_corr.abs() <  CORR_THRESHOLD].index.tolist()

print(f"Informative ({len(informative)}): {informative}")
print(f"Dropped     ({len(dropped)}):     {dropped}")

## 6. Low-Variance Check

Among informative sensors, flag any with low absolute std — high correlation despite low spread warrants inspection.

In [ ]:
low_var = quality.find_low_variance_sensors(train_df[informative], tol=1e-2)

if low_var:
    print(f"Low-variance but informative sensors: {low_var}")
    print(train_df[low_var].describe().round(6))
else:
    print("No low-variance sensors among informative set.")

## 7. Sensor Distributions

In [ ]:
n_cols = 3
n_rows = max(1, len(informative) // n_cols + (1 if len(informative) % n_cols else 0))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(informative):
    train_df[col].hist(bins=50, ax=axes[i], edgecolor="black", alpha=0.7)
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Sensor Distributions (informative sensors)", y=1.02)
plt.tight_layout()
plt.show()

## 8. Degradation Trajectories

Raw and smoothed signals for sample engines, top 4 sensors by |corr with RUL|.

In [ ]:
sample_engines = sorted(train_df["engine_id"].unique())[:5]
sample_df = train_df[train_df["engine_id"].isin(sample_engines)]

top_sensors = (
    rul_corr.abs()
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        d = sample_df[sample_df["engine_id"] == eid]
        ax.plot(d["cycle"], d[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(sensor)
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Raw Sensor Degradation (sample engines)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
smoothed = degradation.compute_sensor_trends(sample_df, top_sensors, window=10)

fig, axes = plt.subplots(len(top_sensors), 1, figsize=(14, 4 * len(top_sensors)))
if len(top_sensors) == 1:
    axes = [axes]
for ax, sensor in zip(axes, top_sensors):
    for eid in sample_engines:
        d = smoothed[smoothed["engine_id"] == eid]
        ax.plot(d["cycle"], d[sensor], alpha=0.7, label=f"Engine {eid}")
    ax.set_ylabel(f"{sensor} (smoothed)")
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("Cycle")
plt.suptitle("Smoothed Sensor Trends (window=10)", y=1.01)
plt.tight_layout()
plt.show()

## 9. Summary

**Dataset:** 24,720 rows, 100 engines, 1 operating condition

**Data quality:** No missing values. Constant sensors (3): `s_1`, `s_18`, `s_19`.

**Engine lifetimes:** min 145, median 220, max 525 cycles — longer-lived than FD001.

**Sensor filter (|corr with RUL| ≥ 0.1):**
- Informative (12): `s_2`, `s_3`, `s_4`, `s_7`, `s_8`, `s_9`, `s_10`, `s_11`, `s_12`, `s_13`, `s_14`, `s_17`
- Dropped (6): `s_5`, `s_6`, `s_15`, `s_16`, `s_20`, `s_21`

**Low-variance check:** `s_10` (std ≈ 0.003) passes correlation (−049 with RUL) but has very low absolute spread — signal is real but near the noise floor.

**No normalization needed** — single operating condition.
